# Grid Search — Two-Stage Predictor

Exploración secuencial en dos etapas:

**Etapa 1 — Clasificador** (¿llueve mañana?): se barren configuraciones de `LGBMClassifier`. El umbral óptimo se busca en validación interna de cada fold (sin tocar test). Métrica principal: **avg_F1**.

**Etapa 2 — Regresor** (¿cuánto llueve?): fijando el mejor clf, se barren configuraciones de `LGBMRegressor` entrenado solo en días lluviosos. Métrica principal: **avg_RMSE_rain** + **avg_Recall_picos**.

**Evaluación final**: la combinación ganadora clf+reg se evalúa en test holdout vs baseline `regression_l1` (mejor modelo del grid previo).

> MLflow: todos los runs de clf llevan `tags.stage = "clf_grid"`, los de reg `tags.stage = "reg_grid"`, y el run final `tags.stage = "two_stage_final"`.

## 1. Setup

In [0]:
%pip install lightgbm --quiet
dbutils.library.restartPython()

In [0]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('/Workspace/Repos/carlos.saquel@gmail.com/santiago-weather-forecast')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data.ingestion import load_from_delta_table
from src.data.preprocessing import build_features, train_test_split, get_feature_names
from src.models.lightgbm_model import LightGBMPredictor
from src.models.two_stage_model import TwoStagePredictor
from src.evaluation.metrics import evaluate_model
from src.evaluation.cross_validation import TimeSeriesSplit
from src.evaluation.two_stage_cv import (
    run_cv_clf_grid, run_cv_reg_grid,
    evaluate_two_stage, find_best_threshold,
)
from src.utils.mlflow_utils import setup_experiment, log_test_evaluation
from src.utils.config import *

setup_experiment()
print('✅ Setup completo')

## 2. Cargar datos

In [0]:
df_raw = load_from_delta_table('weather_raw', spark)
df = build_features(df_raw)
df_train, df_test = train_test_split(df)

print(f'\n📊 Train: {len(df_train)} días  '
      f'({df_train.index.min().date()} → {df_train.index.max().date()})')
print(f'📊 Test:  {len(df_test)} días  '
      f'({df_test.index.min().date()} → {df_test.index.max().date()})')
print(f'📊 Features: {len(get_feature_names(df_train))}')
print(f'\n📊 Días con lluvia (train): '
      f'{(df_train[TARGET] > RAIN_THRESHOLD).sum()} '
      f'({(df_train[TARGET] > RAIN_THRESHOLD).mean()*100:.1f}%)')
print(f'📊 Picos >20mm (train): {(df_train[TARGET] > 20).sum()}')

## 3. Grid Search — Clasificador

Dimensiones exploradas:
- `is_unbalance` vs `scale_pos_weight` para manejar el desbalance 89/11
- `boosting_type`: gbdt vs dart vs goss
- `max_depth` / `num_leaves`: capacidad del modelo
- `learning_rate` + `n_estimators`: velocidad de aprendizaje
- `min_child_samples`: regularización implícita en clases minoritarias

Métrica de selección: **avg_F1** (promedio de folds). El umbral óptimo se busca en validación interna de cada fold para no contaminar el test.

In [0]:
CLF_GRID = [

    # ── GRUPO A: Manejo del desbalance ────────────────────────────
    # Con clf_rain_threshold=0.5mm el ratio neg/pos es ~4:1 (no 8:1)
    {
        'name': 'clf_unbalance_base',
        # Baseline: is_unbalance compensa automáticamente el ratio de clases
        'n_estimators': 300, 'learning_rate': 0.05,
        'max_depth': 6, 'num_leaves': 31,
        'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.8,
    },
    {
        'name': 'clf_scale_pos_weight_4',
        # Ratio calibrado al threshold 0.5mm: ~4 secos por cada lluvioso
        'is_unbalance': False, 'scale_pos_weight': 4.0,
        'n_estimators': 300, 'learning_rate': 0.05,
        'max_depth': 6, 'num_leaves': 31,
        'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.8,
    },
    {
        'name': 'clf_scale_pos_weight_6',
        # Ratio intermedio — más agresivo que 4 pero menos que is_unbalance
        'is_unbalance': False, 'scale_pos_weight': 6.0,
        'n_estimators': 300, 'learning_rate': 0.05,
        'max_depth': 6, 'num_leaves': 31,
        'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.8,
    },

    # ── GRUPO B: Boosting type ────────────────────────────────────
    {
        'name': 'clf_dart_unbalance',
        # DART con is_unbalance: dropout + compensación de desbalance
        'boosting_type': 'dart',
        'n_estimators': 300, 'learning_rate': 0.1,
        'num_leaves': 40, 'drop_rate': 0.15,
        'min_child_samples': 20,
    },
    {
        'name': 'clf_dart_spw4',
        # DART con scale_pos_weight calibrado — más fino que is_unbalance
        'boosting_type': 'dart',
        'is_unbalance': False, 'scale_pos_weight': 4.0,
        'n_estimators': 300, 'learning_rate': 0.1,
        'num_leaves': 40, 'drop_rate': 0.15,
        'min_child_samples': 20,
    },
    {
        'name': 'clf_goss_top20',
        # GOSS con top_rate bajo — más foco en gradientes extremos
        'boosting_type': 'goss',
        'n_estimators': 300, 'learning_rate': 0.05,
        'num_leaves': 31, 'top_rate': 0.2, 'other_rate': 0.1,
        'min_child_samples': 20,
    },
    {
        'name': 'clf_goss_top40',
        # GOSS con top_rate alto — balance entre gradientes altos y bajos
        'boosting_type': 'goss',
        'n_estimators': 300, 'learning_rate': 0.05,
        'num_leaves': 31, 'top_rate': 0.4, 'other_rate': 0.15,
        'min_child_samples': 20,
    },

    # ── GRUPO C: Profundidad ──────────────────────────────────────
    {
        'name': 'clf_shallow',
        # Árbol superficial — menos sobreajuste, mejor generalización temporal
        'n_estimators': 300, 'learning_rate': 0.05,
        'max_depth': 4, 'num_leaves': 15,
        'min_child_samples': 30, 'subsample': 0.8, 'colsample_bytree': 0.8,
    },
    {
        'name': 'clf_deep',
        # Árbol profundo — captura interacciones climatológicas complejas
        'n_estimators': 300, 'learning_rate': 0.05,
        'max_depth': 10, 'num_leaves': 100,
        'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.8,
    },
    {
        'name': 'clf_deep_conservative',
        # Árbol profundo pero con min_child_samples alto — evita sobreajustar folds atípicos
        'n_estimators': 300, 'learning_rate': 0.05,
        'max_depth': 10, 'num_leaves': 100,
        'min_child_samples': 50, 'subsample': 0.8, 'colsample_bytree': 0.8,
    },

    # ── GRUPO D: Velocidad de aprendizaje ─────────────────────────
    {
        'name': 'clf_slow_learner',
        # Aprendizaje fino — más estimadores, menor tasa
        'n_estimators': 800, 'learning_rate': 0.01,
        'max_depth': 6, 'num_leaves': 31,
        'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.8,
    },
    {
        'name': 'clf_fast_shallow',
        # Aprendizaje rápido con árbol chico — baseline rápido competitivo
        'n_estimators': 500, 'learning_rate': 0.08,
        'max_depth': 4, 'num_leaves': 15,
        'min_child_samples': 25, 'subsample': 0.8, 'colsample_bytree': 0.8,
    },

    # ── GRUPO E: Regularización ───────────────────────────────────
    {
        'name': 'clf_high_reg',
        # Alta regularización L1+L2 — robustez ante features ruidosas
        'n_estimators': 300, 'learning_rate': 0.05,
        'max_depth': 6, 'num_leaves': 31,
        'min_child_samples': 30,
        'reg_alpha': 1.0, 'reg_lambda': 5.0,
        'subsample': 0.8, 'colsample_bytree': 0.8,
    },
    {
        'name': 'clf_subsample_aggressive',
        # Sub-muestreo agresivo — reduce sobreajuste con autocorrelación temporal
        'n_estimators': 400, 'learning_rate': 0.05,
        'max_depth': 6, 'num_leaves': 31,
        'min_child_samples': 20,
        'subsample': 0.5, 'colsample_bytree': 0.5, 'subsample_freq': 1,
    },
    {
        'name': 'clf_high_min_child',
        # min_child_samples alto — cada hoja requiere muchos días, más estable entre folds
        'n_estimators': 400, 'learning_rate': 0.05,
        'max_depth': 6, 'num_leaves': 31,
        'min_child_samples': 60,
        'subsample': 0.8, 'colsample_bytree': 0.8,
    },

    # ── GRUPO F: Combinaciones ────────────────────────────────────
    {
        'name': 'clf_combo_shallow_spw4_reg',
        # Árbol superficial + scale_pos_weight calibrado + regularización
        'is_unbalance': False, 'scale_pos_weight': 4.0,
        'n_estimators': 500, 'learning_rate': 0.03,
        'max_depth': 4, 'num_leaves': 15,
        'min_child_samples': 30,
        'reg_alpha': 0.5, 'reg_lambda': 2.0,
        'subsample': 0.8, 'colsample_bytree': 0.8,
    },
    {
        'name': 'clf_combo_dart_deep_spw4',
        # DART profundo con scale_pos_weight calibrado
        'boosting_type': 'dart',
        'is_unbalance': False, 'scale_pos_weight': 4.0,
        'n_estimators': 400, 'learning_rate': 0.05,
        'max_depth': 8, 'num_leaves': 63,
        'drop_rate': 0.1, 'min_child_samples': 20,
    },
    {
        'name': 'clf_combo_slow_reg_spw4',
        # Slow learner + regularización + scale_pos_weight calibrado
        'is_unbalance': False, 'scale_pos_weight': 4.0,
        'n_estimators': 700, 'learning_rate': 0.02,
        'max_depth': 6, 'num_leaves': 31,
        'min_child_samples': 30,
        'reg_alpha': 0.5, 'reg_lambda': 3.0,
        'subsample': 0.8, 'colsample_bytree': 0.8,
    },
]

print(f'📋 Configuraciones clf: {len(CLF_GRID)}')
for c in CLF_GRID:
    print(f'  - {c["name"]}')

In [0]:
results_clf_grid = run_cv_clf_grid(
    clf_grid=CLF_GRID,
    df=df_train,
    thresholds=CLF_THRESHOLDS,
    n_splits=N_SPLITS,
    test_size=TEST_SIZE,
    min_train_size=MIN_TRAIN_SIZE,
    log_mlflow=True,
)

display(results_clf_grid[['name', 'best_threshold', 'avg_auc',
                            'avg_fbeta', 'std_fbeta', 'avg_precision', 'avg_recall']])

In [0]:
# ── Tabla ordenada ────────────────────────────────────────────────
print('\n' + '='*70)
print('RESULTADOS GRID SEARCH — CLASIFICADOR (ordenado por avg_F1)')
print('='*70)
display(
    results_clf_grid[['name', 'best_threshold', 'avg_auc',
                       'avg_fbeta', 'std_fbeta', 'avg_precision', 'avg_recall']]
    .reset_index(drop=True)
)

# ── Barplot comparativo ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.suptitle('Grid Search — Clasificador', fontsize=14, fontweight='bold')

for ax, (metric, title, asc) in zip(axes, [
    ('avg_fbeta',        'avg Fbeta',  False),
    ('avg_precision', 'avg Precision', False),
    ('avg_recall',    'avg Recall',    False),
]):
    df_plot = results_clf_grid.sort_values(metric, ascending=asc)
    colors  = ['#e63946' if i == 0 else 'steelblue' for i in range(len(df_plot))]
    bars = ax.barh(df_plot['name'], df_plot[metric],
                   color=colors, edgecolor='white', alpha=0.85)
    ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=8)
    # Errorbars
    ax.errorbar(df_plot[metric], range(len(df_plot)),
                xerr=df_plot.get(f'std_{metric.replace("avg_","")}',
                                  pd.Series([0]*len(df_plot))),
                fmt='none', color='gray', capsize=3, linewidth=1)
    ax.set_title(title, fontweight='bold')
    ax.grid(alpha=0.3, axis='x')
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

In [0]:
results_clf_grid['score_fbeta'] = results_clf_grid['avg_fbeta'] - results_clf_grid['std_fbeta']
results_clf_grid = results_clf_grid.sort_values('score_fbeta', ascending=False)
best_clf_row       = results_clf_grid.iloc[0]
BEST_CLF_NAME      = best_clf_row['name']
BEST_CLF_THRESHOLD = float(best_clf_row['best_threshold'])

best_cfg        = next(c for c in CLF_GRID if c['name'] == BEST_CLF_NAME)
BEST_CLF_PARAMS = {k: v for k, v in best_cfg.items() if k != 'name'}

print(f'🏆 Mejor clasificador: {BEST_CLF_NAME}')
print(f'   avg_Fbeta: {best_clf_row["avg_fbeta"]:.3f}  (std={best_clf_row["std_fbeta"]:.3f})')
print(f'   avg_AUC:   {best_clf_row["avg_auc"]:.3f}')
print(f'   avg_Prec:  {best_clf_row["avg_precision"]:.3f}')
print(f'   avg_Rec:   {best_clf_row["avg_recall"]:.3f}')
print(f'   Umbral:    {BEST_CLF_THRESHOLD}')
print(f'\n   Params: {BEST_CLF_PARAMS}')

## 4. Grid Search — Regresor

Fijando el mejor clasificador, se barren configuraciones del regresor. El regresor se entrena **solo en días lluviosos** de cada fold de train, y se evalúa en días lluviosos del test de ese fold.

Dimensiones exploradas:
- `objective`: tweedie vs regression_l1 vs quantile
- `tweedie_variance_power`: peso en la cola vs en la ocurrencia
- `log_target`: si comprimir el target con log1p antes de entrenar
- `max_depth` / `num_leaves`: capacidad del modelo
- `reg_alpha` / `reg_lambda`: regularización

Métricas de selección: **avg_RMSE_rain** (principal) + **avg_Recall_picos**.

In [0]:
REG_GRID = [

    # ── GRUPO A: Objetivo — distribución del target ───────────────
    # El target (mm lluvia) tiene cola larga y muchos valores bajos
    # Todos con log_target=True salvo reg_tweedie_1.5_nolog
    {
        'name': 'reg_l1',
        # Baseline robusto — minimiza MAE, no explota con picos extremos
        'objective': 'regression_l1',
        'n_estimators': 300, 'learning_rate': 0.05,
        'max_depth': 6, 'num_leaves': 31,
        'min_child_samples': 15, 'subsample': 0.8, 'colsample_bytree': 0.8,
    },
    {
        'name': 'reg_tweedie_1.2',
        # Tweedie cerca de Poisson — más peso en la ocurrencia (valores bajos)
        'objective': 'tweedie', 'tweedie_variance_power': 1.2,
        'n_estimators': 300, 'learning_rate': 0.05,
        'max_depth': 6, 'num_leaves': 31,
        'min_child_samples': 15, 'subsample': 0.8, 'colsample_bytree': 0.8,
    },
    {
        'name': 'reg_tweedie_1.5',
        # Tweedie moderado — balance ocurrencia/magnitud
        'objective': 'tweedie', 'tweedie_variance_power': 1.5,
        'n_estimators': 300, 'learning_rate': 0.05,
        'max_depth': 6, 'num_leaves': 31,
        'min_child_samples': 15, 'subsample': 0.8, 'colsample_bytree': 0.8,
    },
    {
        'name': 'reg_tweedie_1.8',
        # Tweedie cerca de Gamma — más peso en picos extremos
        'objective': 'tweedie', 'tweedie_variance_power': 1.8,
        'n_estimators': 300, 'learning_rate': 0.05,
        'max_depth': 6, 'num_leaves': 31,
        'min_child_samples': 15, 'subsample': 0.8, 'colsample_bytree': 0.8,
    },
    {
        'name': 'reg_huber',
        # Huber: cuadrático en errores pequeños, lineal en grandes — mejor que L1/L2 puro
        'objective': 'huber', 'alpha': 0.9,
        'n_estimators': 300, 'learning_rate': 0.05,
        'max_depth': 6, 'num_leaves': 31,
        'min_child_samples': 15, 'subsample': 0.8, 'colsample_bytree': 0.8,
    },

    # ── GRUPO B: Quantile — apunta directamente a percentiles ─────
    # No predice la media sino un cuantil: útil para capturar picos
    {
        'name': 'reg_quantile_75',
        # Percentil 75 — sobreestima moderado, captura eventos medios
        'objective': 'quantile', 'alpha': 0.75,
        'n_estimators': 300, 'learning_rate': 0.05,
        'max_depth': 6, 'num_leaves': 31,
        'min_child_samples': 15,
    },
    {
        'name': 'reg_quantile_85',
        # Percentil 85 — más agresivo en picos
        'objective': 'quantile', 'alpha': 0.85,
        'n_estimators': 300, 'learning_rate': 0.05,
        'max_depth': 6, 'num_leaves': 31,
        'min_child_samples': 15,
    },
    {
        'name': 'reg_quantile_90',
        # Percentil 90 — apunta a eventos severos (>20mm)
        'objective': 'quantile', 'alpha': 0.90,
        'n_estimators': 300, 'learning_rate': 0.05,
        'max_depth': 6, 'num_leaves': 31,
        'min_child_samples': 15,
    },

    # ── GRUPO C: Profundidad ──────────────────────────────────────
    {
        'name': 'reg_shallow',
        # Árbol superficial — menos sobreajuste con pocas muestras lluviosas por fold
        'objective': 'tweedie', 'tweedie_variance_power': 1.5,
        'n_estimators': 400, 'learning_rate': 0.05,
        'max_depth': 4, 'num_leaves': 15,
        'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.8,
    },
    {
        'name': 'reg_deep',
        # Árbol profundo — captura relaciones no-lineales entre lags y magnitud
        'objective': 'tweedie', 'tweedie_variance_power': 1.5,
        'n_estimators': 300, 'learning_rate': 0.05,
        'max_depth': 10, 'num_leaves': 100,
        'min_child_samples': 15, 'subsample': 0.8, 'colsample_bytree': 0.8,
    },

    # ── GRUPO D: Regularización ───────────────────────────────────
    {
        'name': 'reg_high_reg_l2',
        # L2 alto — shrinkage suave, útil cuando hay pocos días lluviosos por fold
        'objective': 'tweedie', 'tweedie_variance_power': 1.5,
        'n_estimators': 300, 'learning_rate': 0.05,
        'max_depth': 6, 'num_leaves': 31,
        'min_child_samples': 20,
        'reg_alpha': 0.0, 'reg_lambda': 10.0,
        'subsample': 0.8, 'colsample_bytree': 0.8,
    },
    {
        'name': 'reg_high_reg_l1l2',
        # L1+L2 — sparsity + shrinkage
        'objective': 'tweedie', 'tweedie_variance_power': 1.5,
        'n_estimators': 300, 'learning_rate': 0.05,
        'max_depth': 6, 'num_leaves': 31,
        'min_child_samples': 20,
        'reg_alpha': 1.0, 'reg_lambda': 5.0,
        'subsample': 0.8, 'colsample_bytree': 0.8,
    },
    {
        'name': 'reg_high_min_child',
        # min_child_samples alto — cada hoja requiere muchos días lluviosos
        'objective': 'tweedie', 'tweedie_variance_power': 1.5,
        'n_estimators': 300, 'learning_rate': 0.05,
        'max_depth': 6, 'num_leaves': 31,
        'min_child_samples': 40,
        'subsample': 0.8, 'colsample_bytree': 0.8,
    },

    # ── GRUPO E: Slow learner ─────────────────────────────────────
    {
        'name': 'reg_slow_tweedie',
        # Slow learner con tweedie — ajuste fino de la distribución
        'objective': 'tweedie', 'tweedie_variance_power': 1.5,
        'n_estimators': 800, 'learning_rate': 0.01,
        'max_depth': 6, 'num_leaves': 31,
        'min_child_samples': 15, 'subsample': 0.8, 'colsample_bytree': 0.8,
    },
    {
        'name': 'reg_slow_l1',
        # Slow learner con L1 — robustez + convergencia fina
        'objective': 'regression_l1',
        'n_estimators': 800, 'learning_rate': 0.01,
        'max_depth': 6, 'num_leaves': 31,
        'min_child_samples': 15, 'subsample': 0.8, 'colsample_bytree': 0.8,
    },

    # ── GRUPO F: Referencia sin log_target ───────────────────────
    {
        'name': 'reg_tweedie_1.5_nolog',
        # Igual que tweedie_1.5 pero sin transformación log — referencia cruzada
        'objective': 'tweedie', 'tweedie_variance_power': 1.5,
        'n_estimators': 300, 'learning_rate': 0.05,
        'max_depth': 6, 'num_leaves': 31,
        'min_child_samples': 15, 'subsample': 0.8, 'colsample_bytree': 0.8,
    },
    {
        'name': 'reg_l1_nolog',
        # L1 sin log — referencia cruzada para L1
        'objective': 'regression_l1',
        'n_estimators': 300, 'learning_rate': 0.05,
        'max_depth': 6, 'num_leaves': 31,
        'min_child_samples': 15, 'subsample': 0.8, 'colsample_bytree': 0.8,
    },

    # ── GRUPO G: Combinaciones ────────────────────────────────────
    {
        'name': 'reg_combo_tweedie18_reg_slow',
        # Tweedie agresivo + regularización + slow learner — apuesta por picos
        'objective': 'tweedie', 'tweedie_variance_power': 1.8,
        'n_estimators': 700, 'learning_rate': 0.02,
        'max_depth': 6, 'num_leaves': 31,
        'min_child_samples': 15,
        'reg_alpha': 0.5, 'reg_lambda': 3.0,
        'subsample': 0.8, 'colsample_bytree': 0.8,
    },
    {
        'name': 'reg_combo_l1_shallow_reg',
        # L1 superficial + regularización — máxima robustez con pocas muestras
        'objective': 'regression_l1',
        'n_estimators': 500, 'learning_rate': 0.03,
        'max_depth': 4, 'num_leaves': 15,
        'min_child_samples': 20,
        'reg_alpha': 0.5, 'reg_lambda': 3.0,
        'subsample': 0.8, 'colsample_bytree': 0.8,
    },
    {
        'name': 'reg_combo_q85_deep_reg',
        # Quantile 85 profundo + regularización — captura picos con control de sobreajuste
        'objective': 'quantile', 'alpha': 0.85,
        'n_estimators': 400, 'learning_rate': 0.04,
        'max_depth': 8, 'num_leaves': 63,
        'min_child_samples': 15,
        'reg_alpha': 0.5, 'reg_lambda': 2.0,
        'subsample': 0.8, 'colsample_bytree': 0.8,
    },
]

print(f'📋 Configuraciones reg: {len(REG_GRID)}')
for r in REG_GRID:
    print(f'  - {r["name"]}')
print(f'\n  Clf fijo: {BEST_CLF_NAME}  |  Umbral fijo: {BEST_CLF_THRESHOLD}')

In [0]:
# Nota: reg_tweedie_1.5_nolog se corre aparte con log_target=False
REG_GRID_LOG    = [r for r in REG_GRID if r['name'] != 'reg_tweedie_1.5_nolog']
REG_GRID_NOLOG  = [r for r in REG_GRID if r['name'] == 'reg_tweedie_1.5_nolog']

results_reg_log = run_cv_reg_grid(
    reg_grid=REG_GRID_LOG,
    best_clf_params=BEST_CLF_PARAMS,
    best_threshold=BEST_CLF_THRESHOLD,
    df=df_train,
    log_target=True,
    n_splits=N_SPLITS,
    test_size=TEST_SIZE,
    min_train_size=MIN_TRAIN_SIZE,
    log_mlflow=True,
)

results_reg_nolog = run_cv_reg_grid(
    reg_grid=REG_GRID_NOLOG,
    best_clf_params=BEST_CLF_PARAMS,
    best_threshold=BEST_CLF_THRESHOLD,
    df=df_train,
    log_target=False,
    n_splits=N_SPLITS,
    test_size=TEST_SIZE,
    min_train_size=MIN_TRAIN_SIZE,
    log_mlflow=True,
)

results_reg_grid = pd.concat([results_reg_log, results_reg_nolog], ignore_index=True)
results_reg_grid = results_reg_grid.sort_values('avg_rmse_rain', ascending=True)

display(results_reg_grid[['name', 'log_target', 'avg_mae_rain', 'avg_rmse_rain',
                            'std_rmse_rain', 'avg_r2_rain', 'avg_recall_picos', 'avg_rmse']])

In [0]:
print('\n' + '='*70)
print('RESULTADOS GRID SEARCH — REGRESOR (ordenado por avg_RMSE_rain)')
print('='*70)
display(
    results_reg_grid[['name', 'log_target', 'avg_mae_rain', 'avg_rmse_rain',
                       'std_rmse_rain', 'avg_r2_rain', 'avg_recall_picos']]
    .reset_index(drop=True)
)

# ── Barplot comparativo ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.suptitle('Grid Search — Regresor', fontsize=14, fontweight='bold')

for ax, (metric, title, asc) in zip(axes, [
    ('avg_rmse_rain',    'avg RMSE (días lluvia)',  True),
    ('avg_r2_rain',      'avg R² (días lluvia)',    False),
    ('avg_recall_picos', 'avg Recall picos >20mm',  False),
]):
    df_plot = results_reg_grid.sort_values(metric, ascending=asc).dropna(subset=[metric])
    colors  = ['#e63946' if i == 0 else 'steelblue' for i in range(len(df_plot))]
    bars = ax.barh(df_plot['name'], df_plot[metric],
                   color=colors, edgecolor='white', alpha=0.85)
    ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=8)
    ax.set_title(title, fontweight='bold')
    ax.grid(alpha=0.3, axis='x')
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

In [0]:
best_reg_row  = results_reg_grid.iloc[0]
BEST_REG_NAME = best_reg_row['name']
BEST_LOG_TARGET = bool(best_reg_row['log_target'])

best_reg_cfg    = next(r for r in REG_GRID if r['name'] == BEST_REG_NAME)
BEST_REG_PARAMS = {k: v for k, v in best_reg_cfg.items() if k != 'name'}

print(f'🏆 Mejor regresor: {BEST_REG_NAME}')
print(f'   avg_RMSE_rain:    {best_reg_row["avg_rmse_rain"]:.2f} mm')
print(f'   avg_R²_rain:      {best_reg_row["avg_r2_rain"]:.3f}')
print(f'   avg_Recall_picos: {best_reg_row["avg_recall_picos"]*100:.0f}%')
print(f'   log_target:       {BEST_LOG_TARGET}')
print(f'\n   Params: {BEST_REG_PARAMS}')

## 5. Evaluación final — Test Set

Reentrenar la combinación ganadora con `df_train` completo y evaluar en test holdout. Comparar contra el baseline `regression_l1` (mejor modelo del grid previo de single-stage).

In [0]:
# ── Baseline: modelo único regression_l1 ─────────────────────────
BASELINE_PARAMS = {
    'objective':         'regression_l1',
    'n_estimators':      300,
    'learning_rate':     0.05,
    'max_depth':         6,
    'num_leaves':        31,
    'min_child_samples': 20,
    'subsample':         0.8,
    'colsample_bytree':  0.8,
}
baseline = LightGBMPredictor(**BASELINE_PARAMS)
baseline.fit(df_train)
preds_baseline = baseline.predict(df_test)

# ── Two-stage ganador ─────────────────────────────────────────────
ts_final = TwoStagePredictor(
    clf_params=BEST_CLF_PARAMS,
    reg_params=BEST_REG_PARAMS,
    threshold=BEST_CLF_THRESHOLD,
    log_target=BEST_LOG_TARGET,
)
ts_final.fit(df_train)
preds_ts = ts_final.predict(df_test)
prob_ts  = ts_final.predict_proba(df_test)

y_test = df_test[TARGET]
print(f'✅ Ambos modelos entrenados con {len(df_train)} días')
print(f'   Test: {len(df_test)} días')

In [0]:
metrics_baseline = evaluate_model(y_test, preds_baseline, model_name='regression_l1')
metrics_ts       = evaluate_two_stage(
    y_true=y_test, y_pred=preds_ts, prob_llueve=prob_ts,
    threshold=BEST_CLF_THRESHOLD, label='TwoStage',
)

print('\n' + '='*68)
print('COMPARATIVA FINAL — Test Set')
print('='*68)
print(f'  {"Métrica":24s}  {"Baseline (L1)":>16}  {"Two-Stage":>12}  {"Δ":>8}')
print(f'  {"-"*64}')

comparaciones = [
    ('rmse',         'RMSE (todos)',         True),
    ('r2',           'R² (todos)',           False),
    ('f1_score',     'F1 (clasificación)',   False),
    ('recall',       'Recall lluvia',        False),
    ('recall_picos', 'Recall picos >20mm',   False),
    ('mae_rain',     'MAE (días lluvia)',     True),
    ('rmse_rain',    'RMSE (días lluvia)',    True),
    ('r2_rain',      'R² (días lluvia)',      False),
    ('auc',          'AUC-ROC',              False),
]
for key, label, lower_better in comparaciones:
    v_b  = metrics_baseline.get(key, np.nan)
    v_ts = metrics_ts.get(key, np.nan)
    if pd.isna(v_b) or pd.isna(v_ts):
        print(f'  {label:24s}  {"—":>16}  {"—":>12}')
        continue
    delta = v_ts - v_b
    emoji = '✅' if (delta < 0 and lower_better) or (delta > 0 and not lower_better) else ('🔴' if delta != 0 else '⚪')
    print(f'  {label:24s}  {v_b:>16.3f}  {v_ts:>12.3f}  {emoji} {delta:+.3f}')

In [0]:
mask_inv = (y_test.index.month >= 5) & (y_test.index.month <= 9)
y_inv    = y_test[mask_inv]

fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('Comparativa Test Set — Temporada de Lluvia (May–Sep)',
             fontsize=14, fontweight='bold')

for ax, (preds, label, color) in zip(axes, [
    (preds_baseline[mask_inv], 'Baseline (regression_l1)', '#e63946'),
    (preds_ts[mask_inv],       f'Two-Stage  clf={BEST_CLF_NAME}  reg={BEST_REG_NAME}  thr={BEST_CLF_THRESHOLD:.2f}', '#52b788'),
]):
    ax.fill_between(y_inv.index, 0, y_inv.values,
                    alpha=0.15, color='steelblue', label='Real')
    ax.plot(y_inv.index, y_inv.values,
            color='steelblue', linewidth=1.2, alpha=0.8)
    ax.plot(preds.index, preds.values,
            color=color, linewidth=1.5, alpha=0.9, label=label)
    mask_p = y_inv > 20
    ax.scatter(y_inv.index[mask_p], y_inv.values[mask_p],
               color='navy', s=50, zorder=5, label='Picos reales (>20mm)')
    ax.scatter(preds.index[mask_p], preds.values[mask_p],
               color=color, s=50, marker='x', zorder=5, linewidth=2,
               label='Pred. en picos')
    ax.set_ylabel('mm')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [0]:
imp = ts_final.get_feature_importance(top_n=15)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle(f'Importancia — TwoStage  |  clf: {BEST_CLF_NAME}  |  reg: {BEST_REG_NAME}',
             fontsize=12, fontweight='bold')

for ax, (stage, title) in zip(axes, [
    ('clf', '🎯 Clasificador — ¿Llueve mañana?'),
    ('reg', '📏 Regresor — ¿Cuánto llueve?'),
]):
    df_imp = imp[stage]
    bars = ax.barh(df_imp['feature'][::-1], df_imp['importance_pct'][::-1],
                   color='steelblue', edgecolor='white', alpha=0.85)
    ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
    ax.set_xlabel('Importancia (%)')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print('\n💡 Señales complementarias: si clf y reg priorizan features distintas,')
print('   el two-stage captura dimensiones que un modelo único ignora.')

In [0]:
import mlflow
from src.utils.config import EXPERIMENT_NAME

with mlflow.start_run(run_name=f'two_stage_final__{BEST_CLF_NAME}__{BEST_REG_NAME}'):
    mlflow.set_tag('stage', 'two_stage_final')
    mlflow.set_tag('clf_name', BEST_CLF_NAME)
    mlflow.set_tag('reg_name', BEST_REG_NAME)

    # Params
    for k, v in BEST_CLF_PARAMS.items():
        mlflow.log_param(f'clf_{k}', v)
    for k, v in BEST_REG_PARAMS.items():
        mlflow.log_param(f'reg_{k}', v)
    mlflow.log_param('threshold', BEST_CLF_THRESHOLD)
    mlflow.log_param('log_target', BEST_LOG_TARGET)

    # Métricas test
    for k in ['rmse', 'r2', 'mae', 'f1_score', 'recall',
              'recall_picos', 'mae_rain', 'rmse_rain', 'r2_rain', 'auc']:
        v = metrics_ts.get(k, None)
        if v is not None and not (isinstance(v, float) and np.isnan(v)):
            mlflow.log_metric(f'test_{k}', float(v))

    # Métricas baseline para comparación
    for k in ['rmse', 'r2', 'mae', 'f1_score']:
        v = metrics_baseline.get(k, None)
        if v is not None:
            mlflow.log_metric(f'baseline_{k}', float(v))

    print(f'✅ Run final loggeado en MLflow')
    print(f'   Run ID: {mlflow.active_run().info.run_id}')